## Restaurant Health Inspection Data Cleaning

### 1.Initial the Inspection of Data

In [1]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine

In [3]:
df = pd.read_csv(r'D:\Data Analysis\1.Projects\1.Restaurant Health Inspection Analysis\Resources\DOHMH_New_York_City_Restaurant_Inspection_Results_20250919.csv')

In [5]:
df.head()

,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,RECORD DATE,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location Point1
0,50104876,NOODLE SUPER NO I,Manhattan,265,1 AVENUE,10003.0,2125290539,Chinese,08/24/2022,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,Critical,26.0,NaN,NaN,09/18/2025,Pre-permit (Operational) / Second Compliance I...,40.732264,-73.981768,106.0,2.0,4800.0,1020423.0,1.009220e+09,MN21,NaN
1,50174387,TREATS OF KOREA,Queens,3150,STEINWAY ST,11103.0,8453232965,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,09/18/2025,NaN,40.760128,-73.918066,401.0,22.0,15500.0,4010534.0,4.006580e+09,QN70,NaN
2,50122756,MELLER'S SPORTS HUB & GRILL,Manhattan,1702,2 AVENUE,10128.0,9175964244,American,08/04/2025,Violations were cited in the following area(s).,04L,Evidence of mice or live mice in establishment...,Critical,80.0,NaN,NaN,09/18/2025,Cycle Inspection / Initial Inspection,40.779327,-73.950670,108.0,5.0,14602.0,1050066.0,1.015510e+09,MN32,NaN
3,41408131,CHRIS RESTAURANT,Brooklyn,1866,86 STREET,11214.0,3474623755,Polish,05/20/2024,Violations were cited in the following area(s).,10G,Dishwashing and ware washing: Cleaning and san...,Not Critical,12.0,A,05/20/2024,09/18/2025,Cycle Inspection / Re-inspection,40.606285,-74.001070,311.0,38.0,27800.0,3340602.0,3.063710e+09,BK28,NaN
4,50138007,PEARL OF CHINA,Brooklyn,8411,3 AVENUE,11209.0,7188334281,Chinese,08/16/2023,Violations were cited in the following area(s).,09E,Wash hands sign not posted near or above hand ...,Not Critical,8.0,A,08/16/2023,09/18/2025,Pre-permit (Operational) / Initial Inspection,40.624850,-74.030476,310.0,47.0,6200.0,3152743.0,3.060250e+09,BK31,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288888 entries, 0 to 288887
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  288888 non-null  int64  
 1   DBA                    288882 non-null  object 
 2   BORO                   288888 non-null  object 
 3   BUILDING               288379 non-null  object 
 4   STREET                 288885 non-null  object 
 5   ZIPCODE                285977 non-null  float64
 6   PHONE                  288882 non-null  object 
 7   CUISINE DESCRIPTION    285189 non-null  object 
 8   INSPECTION DATE        288888 non-null  object 
 9   ACTION                 285189 non-null  object 
 10  VIOLATION CODE         283037 non-null  object 
 11  VIOLATION DESCRIPTION  283037 non-null  object 
 12  CRITICAL FLAG          288888 non-null  object 
 13  SCORE                  272962 non-null  float64
 14  GRADE                  140844 non-nu

In [4]:
pd.set_option('display.max.rows', 5000)
pd.set_option('display.max.columns', 27)

### 2.Drop the Duplicates

In [4]:
df.columns = (df.columns
              .str.lower()
              .str.strip()
              .str.replace(' ','_')
             )

In [9]:
df.columns

Index(['camis', 'dba', 'boro', 'building', 'street', 'zipcode', 'phone',
       'cuisine_description', 'inspection_date', 'action', 'violation_code',
       'violation_description', 'critical_flag', 'score', 'grade',
       'grade_date', 'record_date', 'inspection_type', 'latitude', 'longitude',
       'community_board', 'council_district', 'census_tract', 'bin', 'bbl',
       'nta', 'location_point1'],
      dtype='object')

In [15]:
df[['camis','inspection_date','violation_code']].nunique()

camis              30361
inspection_date     1793
violation_code       148
dtype: int64

In [7]:
df.duplicated(subset = ['camis','inspection_date','violation_code']).sum() 

np.int64(21)

In [5]:
df.drop_duplicates(subset = ['camis','inspection_date','violation_code'], inplace = True)
df.shape

(288867, 27)

### 3. Type Conversion

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 288867 entries, 0 to 288887
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   camis                  288867 non-null  int64         
 1   dba                    288861 non-null  object        
 2   boro                   288867 non-null  object        
 3   building               288358 non-null  object        
 4   street                 288864 non-null  object        
 5   zipcode                285962 non-null  float64       
 6   phone                  288861 non-null  object        
 7   cuisine_description    285168 non-null  object        
 8   inspection_date        288867 non-null  datetime64[ns]
 9   action                 285168 non-null  object        
 10  violation_code         283027 non-null  object        
 11  violation_description  283027 non-null  object        
 12  critical_flag          288867 non-null  object   

In [6]:
df['inspection_date'] = pd.to_datetime(df['inspection_date'])
df['grade_date'] = pd.to_datetime(df['grade_date'])
df['record_date'] = pd.to_datetime(df['record_date'])

### 4. Remove Irrelevant Columns

In [10]:
df.columns

Index(['camis', 'dba', 'boro', 'building', 'street', 'zipcode', 'phone',
       'cuisine_description', 'inspection_date', 'action', 'violation_code',
       'violation_description', 'critical_flag', 'score', 'grade',
       'grade_date', 'record_date', 'inspection_type', 'latitude', 'longitude',
       'community_board', 'council_district', 'census_tract', 'bin', 'bbl',
       'nta', 'location_point1'],
      dtype='object')

In [11]:
df.head(3)

,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,violation_code,violation_description,critical_flag,score,grade,grade_date,record_date,inspection_type,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta,location_point1
0,50104876,NOODLE SUPER NO I,Manhattan,265,1 AVENUE,10003.0,2125290539,Chinese,2022-08-24,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,Critical,26.0,NaN,NaT,2025-09-18,Pre-permit (Operational) / Second Compliance I...,40.732264,-73.981768,106.0,2.0,4800.0,1020423.0,1.009220e+09,MN21,NaN
1,50174387,TREATS OF KOREA,Queens,3150,STEINWAY ST,11103.0,8453232965,NaN,1900-01-01,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2025-09-18,NaN,40.760128,-73.918066,401.0,22.0,15500.0,4010534.0,4.006580e+09,QN70,NaN
2,50122756,MELLER'S SPORTS HUB & GRILL,Manhattan,1702,2 AVENUE,10128.0,9175964244,American,2025-08-04,Violations were cited in the following area(s).,04L,Evidence of mice or live mice in establishment...,Critical,80.0,NaN,NaT,2025-09-18,Cycle Inspection / Initial Inspection,40.779327,-73.950670,108.0,5.0,14602.0,1050066.0,1.015510e+09,MN32,NaN


In [7]:
df = df.drop(['building','phone','latitude','longitude','community_board','council_district','census_tract','bin','bbl','nta','location_point1'],axis = 1)
df.head()        

,camis,dba,boro,street,zipcode,cuisine_description,inspection_date,action,violation_code,violation_description,critical_flag,score,grade,grade_date,record_date,inspection_type
0,50104876,NOODLE SUPER NO I,Manhattan,1 AVENUE,10003.0,Chinese,2022-08-24,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,Critical,26.0,NaN,NaT,2025-09-18,Pre-permit (Operational) / Second Compliance I...
1,50174387,TREATS OF KOREA,Queens,STEINWAY ST,11103.0,NaN,1900-01-01,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2025-09-18,NaN
2,50122756,MELLER'S SPORTS HUB & GRILL,Manhattan,2 AVENUE,10128.0,American,2025-08-04,Violations were cited in the following area(s).,04L,Evidence of mice or live mice in establishment...,Critical,80.0,NaN,NaT,2025-09-18,Cycle Inspection / Initial Inspection
3,41408131,CHRIS RESTAURANT,Brooklyn,86 STREET,11214.0,Polish,2024-05-20,Violations were cited in the following area(s).,10G,Dishwashing and ware washing: Cleaning and san...,Not Critical,12.0,A,2024-05-20,2025-09-18,Cycle Inspection / Re-inspection
4,50138007,PEARL OF CHINA,Brooklyn,3 AVENUE,11209.0,Chinese,2023-08-16,Violations were cited in the following area(s).,09E,Wash hands sign not posted near or above hand ...,Not Critical,8.0,A,2023-08-16,2025-09-18,Pre-permit (Operational) / Initial Inspection


### 5. Standardising the Cuisine_Description Columns

In [13]:
df['cuisine_description'].nunique()

89

In [14]:
set(df['cuisine_description'])

{'Afghan',
 'African',
 'American',
 'Armenian',
 'Asian/Asian Fusion',
 'Australian',
 'Bagels/Pretzels',
 'Bakery Products/Desserts',
 'Bangladeshi',
 'Barbecue',
 'Basque',
 'Bottled Beverages',
 'Brazilian',
 'Cajun',
 'Californian',
 'Caribbean',
 'Chicken',
 'Chilean',
 'Chimichurri',
 'Chinese',
 'Chinese/Cuban',
 'Chinese/Japanese',
 'Coffee/Tea',
 'Continental',
 'Creole',
 'Creole/Cajun',
 'Czech',
 'Donuts',
 'Eastern European',
 'Egyptian',
 'English',
 'Ethiopian',
 'Filipino',
 'French',
 'Frozen Desserts',
 'Fruits/Vegetables',
 'Fusion',
 'German',
 'Greek',
 'Hamburgers',
 'Haute Cuisine',
 'Hawaiian',
 'Hotdogs',
 'Hotdogs/Pretzels',
 'Indian',
 'Indonesian',
 'Iranian',
 'Irish',
 'Italian',
 'Japanese',
 'Jewish/Kosher',
 'Juice, Smoothies, Fruit Salads',
 'Korean',
 'Latin American',
 'Lebanese',
 'Mediterranean',
 'Mexican',
 'Middle Eastern',
 'Moroccan',
 'New American',
 'New French',
 'Not Listed/Not Applicable',
 'Nuts/Confectionary',
 'Other',
 'Pakistani',


In [8]:
df['cuisine_description'] = df['cuisine_description'].fillna('Not Listed')

In [9]:
df[df['cuisine_description'].str.contains('Caribbean')]
# most of cuisine named 'Asian' are east Asian cuisine restaurants ,so put the cusisine Asian into 'East Asian' category 

,camis,dba,boro,street,zipcode,cuisine_description,inspection_date,action,violation_code,violation_description,critical_flag,score,grade,grade_date,record_date,inspection_type
18,50156625,EVERFRESH RESTAURANT,Brooklyn,NOSTRAND AVENUE,11226.0,Caribbean,2024-08-12,Violations were cited in the following area(s).,10G,Dishwashing and ware washing: Cleaning and san...,Not Critical,58.0,N,NaT,2025-09-18,Pre-permit (Non-operational) / Initial Inspection
19,50086657,TJ'S TASTY CORNER 2,Brooklyn,RUTLAND ROAD,11212.0,Caribbean,2023-01-18,Establishment Closed by DOHMH. Violations were...,04L,Evidence of mice or live mice in establishment...,Critical,31.0,NaN,NaT,2025-09-18,Cycle Inspection / Re-inspection
33,50089305,CJ CAFE,Queens,S CONDUIT AVE,11422.0,Caribbean,2022-04-14,Violations were cited in the following area(s).,02G,Cold food item held above 41º F (smoked fish a...,Critical,26.0,NaN,NaT,2025-09-18,Cycle Inspection / Initial Inspection
47,50083586,MANDEVILLE'S CUISINE,Brooklyn,FOSTER AVENUE,11236.0,Caribbean,2024-01-03,Violations were cited in the following area(s).,09B,Thawing procedure improper.,Not Critical,12.0,A,2024-01-03,2025-09-18,Cycle Inspection / Initial Inspection
53,50039380,BK9,Brooklyn,5 AVENUE,11217.0,Caribbean,2022-06-08,Violations were cited in the following area(s).,02H,Food not cooled by an approved method whereby ...,Critical,29.0,NaN,NaT,2025-09-18,Cycle Inspection / Initial Inspection
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288811,50071988,HALSEY BAR & GRILL,Brooklyn,AVENUE H,11234.0,Caribbean,2025-08-27,Establishment Closed by DOHMH. Violations were...,08A,Establishment is not free of harborage or cond...,Not Critical,62.0,NaN,NaT,2025-09-18,Cycle Inspection / Initial Inspection
288820,40737045,FISHERMAN'S COVE,Brooklyn,CHURCH AVENUE,11226.0,Caribbean,2022-10-19,Violations were cited in the following area(s).,10D,Mechanical or natural ventilation not provided...,Not Critical,16.0,NaN,NaT,2025-09-18,Cycle Inspection / Initial Inspection
288847,50066640,FISHERMAN'S COVE,Brooklyn,CHURCH AVENUE,11203.0,Caribbean,2025-03-05,Violations were cited in the following area(s).,08A,Establishment is not free of harborage or cond...,Not Critical,20.0,NaN,NaT,2025-09-18,Cycle Inspection / Initial Inspection
288868,50109826,YAH SUH NYCE,Queens,SUTPHIN BLVD,11435.0,Caribbean,2024-07-15,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,Critical,43.0,NaN,NaT,2025-09-18,Cycle Inspection / Initial Inspection


In [10]:
def geographic_cuisine_standardisation(x):
    if x in ['African','Egyptian','Ethiopian','Moroccan']:
        return 'African'
    
    elif x in ['Asian/Asian Fusion','Chinese','Chinese/Cuban','Chinese/Japanese','Japanese','Korean']:
        return 'Northeast Asian'
    
    elif x in ['Armenian','Basque','Continental','Czech','Eastern European','English','French','German','Greek','Irish','Italian',
               'Mediterranean','New French','Polish','Portuguese','Russian','Scandinavian','Spanish','Tapas']:
        return 'European'
    
    elif x in ['Brazilian','Chilean','Chimichurri','Latin American','Peruvian','Mexican','Caribbean']:
        return 'Latin American'
    
    elif x in ['Iranian','Jewish/Kosher','Lebanese','Middle Eastern','Turkish','Afghan']:
        return 'Middle Eastern'
    
    elif x in ['Australian','Hawaiian']:
        return 'Oceanian'
    
    elif x in ['Bagels/Pretzels','Bakery Products/Desserts','Barbecue','Bottled Beverages','Chicken','Coffee/Tea','Donuts','Frozen Desserts',
               'Fruits/Vegetables','Fusion','Haute Cuisine','Hotdogs','Hotdogs/Pretzels','Juice, Smoothies, Fruit Salads',
               'Nuts/Confectionary','Other','Pancakes/Waffles','Salads','Sandwiches','Pizza','Hamburgers',
               'Sandwiches/Salads/Mixed Buffet','Seafood','Soul Food','Soups','Soups/Salads/Sandwiches','Steakhouse','Vegan','Vegetarian']:
        return 'Neutral'

    elif x in ['American','Cajun','Californian','Creole','Creole/Cajun','New American','Southwestern','Tex-Mex']:
        return 'North American'
    
    elif x in ['Bangladeshi','Indian','Pakistani']:
        return 'South Asian'

    elif x in [ 'Filipino','Indonesian','Southeast Asian','Thai']:
        return 'Southeast Asian'
        
    elif x in ['Not Listed/Not Applicable','Not Listed']:
        return 'Not Listed'

    else: 
        return x

df['cuisine_geographic_category'] = df['cuisine_description'].apply(geographic_cuisine_standardisation)



In [11]:
df.groupby('cuisine_geographic_category')['camis'].count()

cuisine_geographic_category
African             1852
European           29874
Latin American     36712
Middle Eastern      7783
Neutral            96520
North American     49832
Northeast Asian    48792
Not Listed          3744
Oceanian             755
South Asian         6568
Southeast Asian     6435
Name: camis, dtype: int64

In [12]:
def thematic_cuisine_categorisation(x):

    if x in ["Hamburgers", "Hotdogs", "Pizza", "Sandwiches", "Donuts",
    "Chicken", "Hotdogs/Pretzels", "Bagels/Pretzels",
    "Pancakes/Waffles", "Sandwiches/Salads/Mixed Buffet", "Soups/Salads/Sandwiches"]:
        return "Quick service"

    elif x in ["Afghan", "African", "American", "Armenian", "Asian/Asian Fusion", "Australian", "Bangladeshi", "Barbecue", "Basque",
                "Brazilian", "Cajun", "Californian", "Caribbean", "Chilean", "Chimichurri", "Chinese", "Chinese/Cuban", "Chinese/Japanese",
                "Continental", "Creole", "Creole/Cajun", "Czech", "Eastern European", "Egyptian", "English", "Ethiopian",
                "Filipino", "German", "Greek", "Hawaiian", "Indian", "Indonesian", "Iranian", "Irish", "Italian", "Japanese",
                "Jewish/Kosher", "Korean", "Latin American", "Lebanese", "Mediterranean", "Mexican", "Middle Eastern",
                "Moroccan", "New American", "New French", "Pakistani", "Peruvian", "Polish", "Portuguese", "Russian",
                "Scandinavian", "Southeast Asian", "Southwestern", "Spanish", "Soul Food", "Tapas", "Tex-Mex", "Thai", "Turkish"]:
        return "Ethnic cuisine"

    elif x in ["Vegan", "Vegetarian", "Salads", "Juice, Smoothies, Fruit Salads", "Fruits/Vegetables"]:
        return "Health/Plant-based"

    elif x in ["Bakery Products/Desserts", "Frozen Desserts", "Nuts/Confectionary"]:
        return "Desserts/Bakery"


    elif x in ["Coffee/Tea", "Bottled Beverages"]:
        return "Beverage-centric"

    elif x in ["Haute Cuisine", "French", "Steakhouse"]:
        return "Fine dining"

    elif x in ["Fusion", "Seafood", "Soups", "Other"]:
        return "Others"
    else:
        return "Not listed"
        
df["thematic_cuisine_category"] = df["cuisine_description"].apply(thematic_cuisine_categorisation)


In [13]:
df.groupby('thematic_cuisine_category')['camis'].count()

thematic_cuisine_category
Beverage-centric       21172
Desserts/Bakery        14485
Ethnic cuisine        186949
Fine dining             3866
Health/Plant-based      7531
Not listed              3744
Others                  6197
Quick service          44923
Name: camis, dtype: int64

### 6. Standardising the Inspection Type

In [15]:
set(df['inspection_type'])

{'Administrative Miscellaneous / Compliance Inspection',
 'Administrative Miscellaneous / Initial Inspection',
 'Administrative Miscellaneous / Re-inspection',
 'Administrative Miscellaneous / Reopening Inspection',
 'Administrative Miscellaneous / Second Compliance Inspection',
 'Calorie Posting / Compliance Inspection',
 'Calorie Posting / Initial Inspection',
 'Calorie Posting / Re-inspection',
 'Cycle Inspection / Compliance Inspection',
 'Cycle Inspection / Initial Inspection',
 'Cycle Inspection / Re-inspection',
 'Cycle Inspection / Reopening Inspection',
 'Cycle Inspection / Second Compliance Inspection',
 'Inter-Agency Task Force / Initial Inspection',
 'Inter-Agency Task Force / Re-inspection',
 'Pre-permit (Non-operational) / Compliance Inspection',
 'Pre-permit (Non-operational) / Initial Inspection',
 'Pre-permit (Non-operational) / Re-inspection',
 'Pre-permit (Non-operational) / Second Compliance Inspection',
 'Pre-permit (Operational) / Compliance Inspection',
 'Pre-per

In [14]:
df[['inspection_program','inspection_type_performed']] = df['inspection_type'].str.split('/',expand = True)

In [15]:
df[['inspection_program','sub_program']] = df['inspection_program'].str.split('(',expand = True)
df = df.drop('sub_program',axis = 1)

In [16]:
df['inspection_program'] = df['inspection_program'].str.strip()
df['inspection_type_performed'] = df['inspection_type_performed'].str.strip()

In [19]:
set(df['inspection_program'])

{'Administrative Miscellaneous',
 'Calorie Posting',
 'Cycle Inspection',
 'Inter-Agency Task Force',
 'Pre-permit',
 'Smoke-Free Air Act',
 'Sodium Warning',
 'Trans Fat',
 nan}

In [20]:
set(df['inspection_type_performed'])

{'Compliance Inspection',
 'Initial Inspection',
 'Limited Inspection',
 'Re-inspection',
 'Reopening Inspection',
 'Second Compliance Inspection',
 nan}

In [21]:
# drop original column
df = df.drop('inspection_type',axis = 1)

### 7. Standardising Violation Code
I referred to the [Food Service Establishment Sanitary Inspection Procedures and Letter Grading](https://codelibrary.amlegal.com/codes/newyorkcity/latest/NYCrules/0-0-0-43593) to categorise the violation columns under their broad umbrellas to make analysis easier. There were 148 distinct violation codes. However, many fell under the same broad categories.

In [22]:
df['violation_code'].nunique()

148

In [23]:
df['violation_description'].nunique()

225

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288888 entries, 0 to 288887
Data columns (total 19 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   camis                        288888 non-null  int64         
 1   dba                          288882 non-null  object        
 2   boro                         288888 non-null  object        
 3   street                       288885 non-null  object        
 4   zipcode                      285977 non-null  float64       
 5   cuisine_description          288888 non-null  object        
 6   inspection_date              288888 non-null  datetime64[ns]
 7   action                       285189 non-null  object        
 8   violation_code               283037 non-null  object        
 9   violation_description        283037 non-null  object        
 10  critical_flag                288888 non-null  object        
 11  score                     

In [18]:
# Mapping of violation code prefixes to categories
violation_category = {
    '02': 'Time and Temperature Control for Safety',
    '03': 'Food Source',
    '04': 'Food Protection & Pest Control',
    '05': 'Facility Design and Construction',
    '06': 'Food Worker Hygiene and Other Food Protection',
    '07': 'Obstruction of Department personnel',
    '08': 'Garbage, Waste Disposal and Pest Management',
    '09': 'Food Protection',
    '10': 'Facility Maintenance',
    '15': 'Smoking and Tobacco Use',
    '16': 'Caloric & Nutritional Information',
    '18': 'Administrative & Posting Violations',
    '19': 'Organic & Inorganic Measures',
    '20': 'Allergy & Health Measures',
}


def classify_violation(x):
    if pd.isna(x):
        return 'No Violation / Not Applicable'
    
    for prefix,category in violation_category.items():
        if x.startswith(prefix):
            return  category
    return 'Others'
        

df['violation_category'] = df['violation_code'].apply(classify_violation)
    

In [19]:
df[df['violation_category'].isna()][['violation_code','violation_category']]

,violation_code,violation_category


In [20]:
df['violation_category'].unique()

array(['Time and Temperature Control for Safety',
       'No Violation / Not Applicable', 'Food Protection & Pest Control',
       'Facility Maintenance', 'Food Protection',
       'Food Worker Hygiene and Other Food Protection',
       'Garbage, Waste Disposal and Pest Management', 'Others',
       'Allergy & Health Measures', 'Administrative & Posting Violations',
       'Food Source', 'Smoking and Tobacco Use',
       'Organic & Inorganic Measures', 'Facility Design and Construction',
       'Obstruction of Department personnel',
       'Caloric & Nutritional Information'], dtype=object)

In [24]:
df.head()

,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,violation_code,violation_description,critical_flag,...,longitude,community_board,council_district,census_tract,bin,bbl,nta,location_point1,cuisine_geographic_category,thematic_cuisine_category,inspection_program,inspection_type_performed,violation_category
0,50104876,NOODLE SUPER NO I,Manhattan,265,1 AVENUE,10003.0,2125290539,Chinese,2022-08-24,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,Critical,...,-73.981768,106.0,2.0,4800.0,1020423.0,1.009220e+09,MN21,NaN,Northeast Asian,Ethnic cuisine,Pre-permit,Second Compliance Inspection,Time and Temperature Control for Safety
1,50174387,TREATS OF KOREA,Queens,3150,STEINWAY ST,11103.0,8453232965,Not Listed,1900-01-01,NaN,NaN,NaN,Not Applicable,...,-73.918066,401.0,22.0,15500.0,4010534.0,4.006580e+09,QN70,NaN,Not Listed,Not listed,NaN,NaN,No Violation / Not Applicable
2,50122756,MELLER'S SPORTS HUB & GRILL,Manhattan,1702,2 AVENUE,10128.0,9175964244,American,2025-08-04,Violations were cited in the following area(s).,04L,Evidence of mice or live mice in establishment...,Critical,...,-73.950670,108.0,5.0,14602.0,1050066.0,1.015510e+09,MN32,NaN,North American,Ethnic cuisine,Cycle Inspection,Initial Inspection,Food Protection & Pest Control
3,41408131,CHRIS RESTAURANT,Brooklyn,1866,86 STREET,11214.0,3474623755,Polish,2024-05-20,Violations were cited in the following area(s).,10G,Dishwashing and ware washing: Cleaning and san...,Not Critical,...,-74.001070,311.0,38.0,27800.0,3340602.0,3.063710e+09,BK28,NaN,European,Ethnic cuisine,Cycle Inspection,Re-inspection,Facility Maintenance
4,50138007,PEARL OF CHINA,Brooklyn,8411,3 AVENUE,11209.0,7188334281,Chinese,2023-08-16,Violations were cited in the following area(s).,09E,Wash hands sign not posted near or above hand ...,Not Critical,...,-74.030476,310.0,47.0,6200.0,3152743.0,3.060250e+09,BK31,NaN,Northeast Asian,Ethnic cuisine,Pre-permit,Initial Inspection,Food Protection


### 8. Working With Null Values

In [21]:
df = df.sort_index(axis = 1)

In [22]:
df = df.drop('action',axis = 1)

In [23]:
df.isna().sum()

boro                                0
camis                               0
critical_flag                       0
cuisine_description                 0
cuisine_geographic_category         0
dba                                 6
grade                          148026
grade_date                     155848
inspection_date                     0
inspection_program               3699
inspection_type                  3699
inspection_type_performed        3699
record_date                         0
score                           15917
street                              3
thematic_cuisine_category           0
violation_category                  0
violation_code                   5840
violation_description            5840
zipcode                          2905
dtype: int64

In [24]:
df['dba'] = df['dba'].fillna('Not Specified')
df['grade'] = df['grade'].fillna('Not Graded')
df['grade_date'] = df['grade_date'].fillna('Not Graded')
df['inspection_program'] = df['inspection_program'].fillna('Not Specified')
df['inspection_type_performed'] = df['inspection_type_performed'].fillna('Not Specified')
df['score'] = df['score'].fillna('no score')
df['violation_code'] = df['violation_code'].fillna('Not Specified')
df['violation_description'] = df['violation_description'].fillna('Not Specified')
df['zipcode'] = df['zipcode'].fillna('Not Specified')
df['violation_code'] = df['violation_code'].fillna('Not Specified')

In [32]:
df.isna().sum()

boro                           0
camis                          0
critical_flag                  0
cuisine_description            0
cuisine_geographic_category    0
dba                            0
grade                          0
grade_date                     0
inspection_date                0
inspection_program             0
inspection_type_performed      0
record_date                    0
score                          0
street                         3
thematic_cuisine_category      0
violation_category             0
violation_code                 0
violation_description          0
zipcode                        0
dtype: int64

In [25]:
df.head()

,boro,camis,critical_flag,cuisine_description,cuisine_geographic_category,dba,grade,grade_date,inspection_date,inspection_program,inspection_type,inspection_type_performed,record_date,score,street,thematic_cuisine_category,violation_category,violation_code,violation_description,zipcode
0,Manhattan,50104876,Critical,Chinese,Northeast Asian,NOODLE SUPER NO I,Not Graded,Not Graded,2022-08-24,Pre-permit,Pre-permit (Operational) / Second Compliance I...,Second Compliance Inspection,2025-09-18,26.0,1 AVENUE,Ethnic cuisine,Time and Temperature Control for Safety,02B,Hot TCS food item not held at or above 140 °F.,10003.0
1,Queens,50174387,Not Applicable,Not Listed,Not Listed,TREATS OF KOREA,Not Graded,Not Graded,1900-01-01,Not Specified,NaN,Not Specified,2025-09-18,no score,STEINWAY ST,Not listed,No Violation / Not Applicable,Not Specified,Not Specified,11103.0
2,Manhattan,50122756,Critical,American,North American,MELLER'S SPORTS HUB & GRILL,Not Graded,Not Graded,2025-08-04,Cycle Inspection,Cycle Inspection / Initial Inspection,Initial Inspection,2025-09-18,80.0,2 AVENUE,Ethnic cuisine,Food Protection & Pest Control,04L,Evidence of mice or live mice in establishment...,10128.0
3,Brooklyn,41408131,Not Critical,Polish,European,CHRIS RESTAURANT,A,2024-05-20 00:00:00,2024-05-20,Cycle Inspection,Cycle Inspection / Re-inspection,Re-inspection,2025-09-18,12.0,86 STREET,Ethnic cuisine,Facility Maintenance,10G,Dishwashing and ware washing: Cleaning and san...,11214.0
4,Brooklyn,50138007,Not Critical,Chinese,Northeast Asian,PEARL OF CHINA,A,2023-08-16 00:00:00,2023-08-16,Pre-permit,Pre-permit (Operational) / Initial Inspection,Initial Inspection,2025-09-18,8.0,3 AVENUE,Ethnic cuisine,Food Protection,09E,Wash hands sign not posted near or above hand ...,11209.0


### 11. Export the data to mysql for further analysis

In [27]:
# Preparing the grounds for export
username = "root"
password = "Sanchahe851225"
host = "localhost"
port = "3306"
database = "restaurant_health_inspection"

# Create connection engine
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")


In [28]:
# Export to sql
df.to_sql(
    name="cleaned_inspection_data",   # Table name
    con=engine,
    if_exists="replace",              # or "append" to add without deleting existing
    index=False,                      # Avoid exporting pandas index
    chunksize=1000                    # Handle large datasets in batches
)

288867

In [26]:
# Export clean data to csv for later use
df.to_csv("cleaned_inspection_data.csv", index = False)